# FM2Edge Phase 1.5 — Oxford-IIIT Pet Colab acceptance
This notebook validates download → breed-domain split → PIDNet-S training → held-out-domain evaluation. Oxford-specific code is disposable and does not enter the core pipeline.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Colab menu: Runtime > Change runtime type > GPU"

## Private GitHub repository access
Before running the next cell, create a fine-grained GitHub token restricted to `tiny-ticop/FM2Edge` with **Contents: Read-only**. In Colab, open the key-shaped **Secrets** panel, add it as `GITHUB_TOKEN`, and enable notebook access. Never paste the token into a code cell. See `docs/COLAB_SETUP.md` for the exact steps.

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from google.colab import userdata

REPO_URL = "https://github.com/tiny-ticop/FM2Edge.git"
BRANCH = "main"
CHECKOUT_DIR = Path("/content/FM2Edge")

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError("Add GITHUB_TOKEN in Colab Secrets and enable notebook access.") from exc
if not github_token:
    raise RuntimeError("The GITHUB_TOKEN Colab Secret is empty.")

authorization = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_environment = os.environ.copy()
git_environment.update(
    {
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {authorization}",
    }
)
try:
    if not (CHECKOUT_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)],
            check=True,
            env=git_environment,
        )
    else:
        subprocess.run(
            ["git", "-C", str(CHECKOUT_DIR), "pull", "--ff-only"],
            check=True,
            env=git_environment,
        )
finally:
    github_token = None
    authorization = None
    git_environment = None

In [ ]:
%cd /content/FM2Edge
%pip install -q -e ".[tensorboard]"

In [ ]:
!python scripts/download_oxford_pet.py --root data/oxford_pet
!python scripts/prepare_oxford_pet.py --root data/oxford_pet --output data/oxford_pet/manifest_smoke.csv --max-breeds 12 --max-samples-per-breed 40 --seed 42
!python scripts/make_splits.py --manifest data/oxford_pet/manifest_smoke.csv --output-dir configs/splits/oxford_pet_smoke --folds 5 --seed 42

In [ ]:
!python scripts/train.py --config configs/experiments/pidnet_s_oxford_pet_smoke.yaml

In [ ]:
# This cell can be rerun without retraining after evaluation-code changes.
!python scripts/evaluate.py --config configs/experiments/pidnet_s_oxford_pet_smoke.yaml

In [ ]:
from pathlib import Path

RUN_DIR = Path("/content/FM2Edge/results/pidnet_s_oxford_pet_smoke/fold_01")
print((RUN_DIR / "metrics" / "summary.json").read_text())

Download a self-contained result archive before the Colab runtime ends. If the browser does not start the download, use the Files panel to download `/content/pidnet_s_oxford_pet_smoke_fold_01.zip`.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    "/content/pidnet_s_oxford_pet_smoke_fold_01",
    "zip",
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name,
)
print(f"created: {archive}")
files.download(archive)

Optional: persist only the self-contained run artifacts to Google Drive.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    source = "/content/FM2Edge/results/pidnet_s_oxford_pet_smoke"
    destination = "/content/drive/MyDrive/FM2Edge_results/pidnet_s_oxford_pet_smoke"
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("saved:", destination)